Wudu stuff


In [163]:
import io
from pathlib import Path
import pandas as pd

DATA = Path("data/Einzelteil")

FILES = {
    "t01": ("Einzelteil_T01.txt", b" | | ", b" "),
    "t02": ("Einzelteil_T02.txt", b"  ", b"\t"),
    "t03": ("Einzelteil_T03.txt", b"|", b"\x0b"),
    "t04": ("Einzelteil_T04.csv", b";", b"\n"),
    "t05": ("Einzelteil_T05.csv", b",", b"\n"),
}

# Bytes, die beim Teilimport vom Dateianfang gelesen werden.
# Muss groß genug sein, damit nrows+1 vollstaendige Zeilen enthalten sind.
CHUNK_BYTES = 2_000_000


def load(key: str, nrows: int | None = 1000) -> pd.DataFrame:
    filename, field_sep, row_sep = FILES[key]
    path = DATA / filename

    # Byte vodoo (replacing field- and row- seperators)
    if nrows is None:
        raw = path.read_bytes()
        raw = raw.replace(field_sep, b"\x01").replace(row_sep, b"\n")
    else:
        with open(path, "rb") as f:
            raw = f.read(CHUNK_BYTES)
        raw = raw.replace(field_sep, b"\x01").replace(row_sep, b"\n")
        lines = raw.split(b"\n")
        if len(lines) < nrows + 2 and path.stat().st_size > CHUNK_BYTES:
            raise ValueError(
                f"{filename}: {CHUNK_BYTES} Bytes reichen fuer {nrows} Zeilen nicht aus, "
                "CHUNK_BYTES erhoehen"
            )
        # letzte, evtl. abgeschnittene Zeile faellt raus
        raw = b"\n".join(lines[: nrows + 1])

    df = pd.read_csv(io.BytesIO(raw), sep="\x01", na_values=["NA"], dtype=str)

    # Combine same columns (.x, .y)
    for col in set(c.removesuffix(".x").removesuffix(".y") for c in df.columns):
        if col + ".x" in df.columns and col + ".y" in df.columns:
            df[col] = df[col + ".x"].combine_first(df[col + ".y"])

    # Standartize id and time columns
    part = key.upper()
    df["Part_ID"] = df.get(f"ID_{part}", df.get("Part_ID"))

    if "Produktionsdatum" not in df.columns and "Produktionsdatum_Origin_01011970" in df.columns:
        df["Produktionsdatum"] = pd.to_datetime("1970-01-01") + pd.to_timedelta(
            df["Produktionsdatum_Origin_01011970"].astype(float), unit="D"
        )

    # Convert datatypes
    df["Produktionsdatum"] = pd.to_datetime(df["Produktionsdatum"])
    df["Fehlerhaft_Datum"] = pd.to_datetime(df["Fehlerhaft_Datum"])
    df["Fehlerhaft_Fahrleistung"] = df["Fehlerhaft_Fahrleistung"].str.replace(",", ".", regex=False).astype(float)

    int_cols = ["Herstellernummer", "Werksnummer", "Fehlerhaft"]
    df[int_cols] = df[int_cols].astype("Int64")

    columns = [
        "Part_ID",
        "Herstellernummer",
        "Werksnummer",
        "Produktionsdatum",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ]

    return df[columns].reset_index(drop=True)

Importing Einzelteile files

In [164]:
# Loading the part files by using our helper function
t01 = load("t01")
t02 = load("t02")
t03 = load("t03")
t04 = load("t04")
t05 = load("t05")




t01 = t01[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t02 = t02[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t03 = t03[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t04 = t04[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
t05 = t05[["Part_ID", "Herstellernummer", "Produktionsdatum","Fehlerhaft"]]
einzelteile = [t01, t02, t03, t04, t05]

#Part type erstellen
t01["Part_Type"] = "T01"
t02["Part_Type"] = "T02"
t03["Part_Type"] = "T03"
t04["Part_Type"] = "T04"
t05["Part_Type"] = "T05"

einzelteile_zusammen = pd.concat([t01, t02, t03, t04, t05],axis=0, ignore_index=True)
part_type_column = einzelteile_zusammen.pop("Part_Type")
einzelteile_zusammen.insert(0, "Part_Type",part_type_column)
#display(t01.head(), einzelteile_zusammen.tail(), t04.head())
display(t05.isnull().sum())

Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
Part_Type           0
dtype: int64

importing komponente data

In [165]:
file_paths = [
    'data/Komponente/Bestandteile_Komponente_K1BE1.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI1.csv',
    'data/Komponente/Bestandteile_Komponente_K1BE2.csv',
    'data/Komponente/Bestandteile_Komponente_K1DI2.csv'
]

engine_dfs = []

# for path in file_paths:
#     df = pd.read_csv(path, sep=';').drop(columns=['Unnamed: 0'])
#     engine_dfs.append(df)


komponente_k1be1 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1BE1.csv", sep=';').drop(columns=['Unnamed: 0'])
komponente_k1di1 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1DI1.csv", sep=';').drop(columns=['Unnamed: 0'])
komponente_k1be2 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1BE2.csv", sep=';').drop(columns=['Unnamed: 0'])
komponente_k1di2 = pd.read_csv("data/Komponente/Bestandteile_Komponente_K1DI2.csv", sep=';').drop(columns=['Unnamed: 0'])




#removing the not needed parts

komponente_k1di1 = komponente_k1di1[["ID_T1", "ID_T2","ID_T5", "ID_K1DI1"]]
komponente_k1be2 = komponente_k1be2[["ID_T1", "ID_T2", "ID_K1BE2"]]
komponente_k1di2 = komponente_k1di2[["ID_T1","ID_T2","ID_K1DI2"]]

display(komponente_k1be1.head())
display(komponente_k1di1.head())
display(komponente_k1be2.head())
display(komponente_k1di2.head())



,ID_T1,ID_T2,ID_T3,ID_T4,ID_K1BE1
0,1-201-2011-45,2-201-2011-161,3-202-2023-14,4-202-2023-20,K1BE1-101-1011-1
1,1-201-2011-429,2-201-2011-239,3-202-2023-16,4-202-2023-51,K1BE1-101-1011-2
2,1-201-2011-399,2-201-2011-220,3-202-2023-46,4-202-2023-93,K1BE1-101-1011-3
3,1-201-2011-335,2-202-2022-463,3-202-2023-149,4-204-2042-18,K1BE1-101-1011-4
4,1-204-2044-188,2-202-2022-675,3-202-2023-152,4-204-2042-40,K1BE1-101-1011-5


,ID_T1,ID_T2,ID_T5,ID_K1DI1
0,1-204-2044-27,2-201-2011-144,5-202-2012-89,K1DI1-101-1041-1
1,1-202-2021-32,2-202-2022-577,5-202-2012-155,K1DI1-101-1041-2
2,1-201-2011-238,2-202-2022-514,5-202-2012-199,K1DI1-101-1041-3
3,1-202-2021-297,2-202-2022-626,5-201-2012-57,K1DI1-101-1041-4
4,1-204-2044-73,2-201-2011-209,5-201-2012-157,K1DI1-101-1041-5


,ID_T1,ID_T2,ID_K1BE2
0,1-202-2021-339,2-202-2022-171,K1BE2-101-1011-1
1,1-201-2011-57,2-201-2011-260,K1BE2-101-1011-2
2,1-202-2021-101,2-202-2022-335,K1BE2-101-1011-3
3,1-204-2044-201,2-201-2011-312,K1BE2-101-1011-4
4,1-201-2011-26,2-202-2022-987,K1BE2-101-1011-5


,ID_T1,ID_T2,ID_K1DI2
0,1-204-2044-183,2-202-2022-551,K1DI2-103-1031-1
1,1-201-2011-246,2-202-2022-738,K1DI2-103-1031-2
2,1-201-2011-152,2-201-2011-181,K1DI2-103-1031-3
3,1-201-2011-134,2-202-2022-561,K1DI2-103-1031-4
4,1-202-2021-437,2-202-2022-724,K1DI2-103-1031-5


Melting the data

In [166]:
#K1BE1
melted_k1be1 = komponente_k1be1.melt(id_vars="ID_K1BE1", var_name="Part_Type", value_name="Part_ID")
melted_k1be1 = melted_k1be1.sort_values(["ID_K1BE1", "Part_Type"])
melted_k1be1 = melted_k1be1.drop(columns=["Part_Type"])
display(melted_k1be1.isna().sum())

#K1DI1
melted_k1di1 = komponente_k1di1.melt(id_vars="ID_K1DI1", var_name="Part_Type", value_name="Part_ID")
melted_k1di1 = melted_k1di1.sort_values(["ID_K1DI1", "Part_Type"])
melted_k1di1 = melted_k1di1.drop(columns=["Part_Type"])
display(melted_k1di1.head())

#K1BE2
melted_k1be2 = komponente_k1be2.melt(id_vars="ID_K1BE2", var_name="Part_Type", value_name="Part_ID")
melted_k1be2 = melted_k1be2.sort_values(["ID_K1BE2", "Part_Type"])
melted_k1be2 = melted_k1be2.drop(columns=["Part_Type"])
display(melted_k1be2.head())

#K1DI2
melted_k1di2 = komponente_k1di2.melt(id_vars="ID_K1DI2", var_name="Part_Type", value_name="Part_ID")
melted_k1di2 = melted_k1di2.sort_values(["ID_K1DI2", "Part_Type"])
melted_k1di2 = melted_k1di2.drop(columns=["Part_Type"])
display(melted_k1di2.head())


ID_K1BE1    0
Part_ID     0
dtype: int64

,ID_K1DI1,Part_ID
0,K1DI1-101-1041-1,1-204-2044-27
1192630,K1DI1-101-1041-1,2-201-2011-144
2385260,K1DI1-101-1041-1,5-202-2012-89
9,K1DI1-101-1041-10,1-201-2011-76
1192639,K1DI1-101-1041-10,2-202-2022-219


,ID_K1BE2,Part_ID
0,K1BE2-101-1011-1,1-202-2021-339
409422,K1BE2-101-1011-1,2-202-2022-171
9,K1BE2-101-1011-10,1-202-2021-71
409431,K1BE2-101-1011-10,2-202-2022-145
435,K1BE2-101-1011-100,1-201-2011-1379


,ID_K1DI2,Part_ID
56,K1DI2-102-1021-1,1-201-2011-352
409478,K1DI2-102-1021-1,2-202-2022-1051
65,K1DI2-102-1021-10,1-202-2021-597
409487,K1DI2-102-1021-10,2-202-2022-1400
211,K1DI2-102-1021-100,1-202-2021-1049


In [167]:

melted_k1be1["Engine Type"] = "K1BE1"

melted_k1di1["Engine Type"] = "K1DI1"

melted_k1be2["Engine Type"] = "K1BE2"

melted_k1di2["Engine Type"] = "K1DI2"

display(einzelteile_zusammen.head(), melted_k1be1.head(), melted_k1di1.head(), melted_k1be2.head(), melted_k1di2.head())
display(einzelteile_zusammen.isna().sum(), melted_k1be1.isna().sum(), melted_k1di1.isna().sum(), melted_k1be2.isna().sum(), melted_k1di2.isna().sum())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft
0,T01,1-201-2011-247,201,2008-11-07,0
1,T01,1-201-2011-429,201,2008-11-07,0
2,T01,1-201-2011-363,201,2008-11-07,1
3,T01,1-201-2011-30,201,2008-11-07,0
4,T01,1-201-2011-72,201,2008-11-07,1


,ID_K1BE1,Part_ID,Engine Type
0,K1BE1-101-1011-1,1-201-2011-45,K1BE1
1192630,K1BE1-101-1011-1,2-201-2011-161,K1BE1
2385260,K1BE1-101-1011-1,3-202-2023-14,K1BE1
3577890,K1BE1-101-1011-1,4-202-2023-20,K1BE1
9,K1BE1-101-1011-10,1-201-2011-37,K1BE1


,ID_K1DI1,Part_ID,Engine Type
0,K1DI1-101-1041-1,1-204-2044-27,K1DI1
1192630,K1DI1-101-1041-1,2-201-2011-144,K1DI1
2385260,K1DI1-101-1041-1,5-202-2012-89,K1DI1
9,K1DI1-101-1041-10,1-201-2011-76,K1DI1
1192639,K1DI1-101-1041-10,2-202-2022-219,K1DI1


,ID_K1BE2,Part_ID,Engine Type
0,K1BE2-101-1011-1,1-202-2021-339,K1BE2
409422,K1BE2-101-1011-1,2-202-2022-171,K1BE2
9,K1BE2-101-1011-10,1-202-2021-71,K1BE2
409431,K1BE2-101-1011-10,2-202-2022-145,K1BE2
435,K1BE2-101-1011-100,1-201-2011-1379,K1BE2


,ID_K1DI2,Part_ID,Engine Type
56,K1DI2-102-1021-1,1-201-2011-352,K1DI2
409478,K1DI2-102-1021-1,2-202-2022-1051,K1DI2
65,K1DI2-102-1021-10,1-202-2021-597,K1DI2
409487,K1DI2-102-1021-10,2-202-2022-1400,K1DI2
211,K1DI2-102-1021-100,1-202-2021-1049,K1DI2


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
dtype: int64

ID_K1BE1       0
Part_ID        0
Engine Type    0
dtype: int64

ID_K1DI1       0
Part_ID        0
Engine Type    0
dtype: int64

ID_K1BE2       0
Part_ID        0
Engine Type    0
dtype: int64

ID_K1DI2       0
Part_ID        0
Engine Type    0
dtype: int64

Merging the df

In [168]:
#K1BE1
merged_k1be1 = pd.merge(einzelteile_zusammen, melted_k1be1, how='inner', on="Part_ID")
merged_k1be1 = merged_k1be1.sort_values(["ID_K1BE1", "Part_Type"])
display(merged_k1be1.head())
display(merged_k1be1.isna().sum())

#K1DI1
merged_k1di1 = pd.merge(einzelteile_zusammen, melted_k1di1, how='inner', on="Part_ID")
merged_k1di1 = merged_k1di1.sort_values(["ID_K1DI1", "Part_Type"])
display(merged_k1di1.head())
display(merged_k1di1.isna().sum())

#K1BE2
merged_k1be2 = pd.merge(einzelteile_zusammen, melted_k1be2, how='inner', on="Part_ID")
merged_k1be2 = merged_k1be2.sort_values(["ID_K1BE2", "Part_Type"])
display(merged_k1be2.head())
display(merged_k1be2.isna().sum())

#K1DI2
merged_k1di2 = pd.merge(einzelteile_zusammen, melted_k1di2, how='inner', on="Part_ID")
merged_k1di2 = merged_k1di2.sort_values(["ID_K1DI2", "Part_Type"])
display(merged_k1di2.head())
display(merged_k1di2.isna().sum())


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1BE1,Engine Type
26,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1
474,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1
1286,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
2286,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
39,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1BE1            0
Engine Type         0
dtype: int64

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1DI1,Engine Type
371,T02,2-201-2011-144,201,2008-11-07,0,K1DI1-101-1041-1,K1DI1
48,T01,1-201-2011-76,201,2008-11-07,0,K1DI1-101-1041-10,K1DI1
546,T05,5-201-2012-127,201,2008-11-07,0,K1DI1-101-1041-100,K1DI1
22,T01,1-201-2011-195,201,2008-11-07,0,K1DI1-101-1041-101,K1DI1
354,T02,2-201-2011-280,201,2008-11-07,0,K1DI1-101-1041-101,K1DI1


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1DI1            0
Engine Type         0
dtype: int64

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1BE2,Engine Type
12,T01,1-201-2011-424,201,2008-11-07,0,K1BE2-101-1011-13,K1BE2
13,T01,1-201-2011-242,201,2008-11-07,1,K1BE2-101-1011-14,K1BE2
14,T01,1-201-2011-215,201,2008-11-07,0,K1BE2-101-1011-19,K1BE2
0,T01,1-201-2011-57,201,2008-11-07,0,K1BE2-101-1011-2,K1BE2
113,T02,2-201-2011-260,201,2008-11-07,0,K1BE2-101-1011-2,K1BE2


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1BE2            0
Engine Type         0
dtype: int64

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,ID_K1DI2,Engine Type
8,T01,1-201-2011-352,201,2008-11-07,0,K1DI2-102-1021-1,K1DI2
118,T02,2-201-2011-86,201,2008-11-07,1,K1DI2-102-1021-100,K1DI2
119,T02,2-201-2011-533,201,2008-11-08,0,K1DI2-102-1021-103,K1DI2
62,T01,1-201-2011-653,201,2008-11-08,0,K1DI2-102-1021-104,K1DI2
120,T02,2-201-2011-375,201,2008-11-08,0,K1DI2-102-1021-105,K1DI2


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
ID_K1DI2            0
Engine Type         0
dtype: int64

Checking if motor is in OM1

In [169]:
# Importing OEM1 data
oem11 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")
oem12 = pd.read_csv("data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv", sep=";").drop(columns=["Unnamed: 0"], errors="ignore")

# Combining the OEM1 Types
oem_combined = pd.concat([oem11, oem12], ignore_index=True)

#using only the needed column
oem_combined = oem_combined["ID_Motor"]

display(oem_combined.head())


0     K1BE1-101-1011-7
1    K1BE1-101-1011-12
2    K1BE1-101-1011-38
3    K1BE1-101-1011-97
4    K1BE1-101-1011-65
Name: ID_Motor, dtype: str

Combining all merged df into one

In [170]:
#renaming the columns before the concat
merged_k1be1 = merged_k1be1.rename(columns={"ID_K1BE1": "Motor_ID"})
merged_k1di1 = merged_k1di1.rename(columns={"ID_K1DI1": "Motor_ID"})
merged_k1be2 = merged_k1be2.rename(columns={"ID_K1BE2": "Motor_ID"})
merged_k1di2 = merged_k1di2.rename(columns={"ID_K1DI2": "Motor_ID"})

display(merged_k1be1.head(), merged_k1di1.head(), merged_k1be2.head(), merged_k1di2.head())

final_df = pd.concat([merged_k1be1, merged_k1di1, merged_k1be2, merged_k1di2], ignore_index=True)
display(final_df.head(), final_df.tail())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
26,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1
474,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1
1286,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
2286,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
39,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
371,T02,2-201-2011-144,201,2008-11-07,0,K1DI1-101-1041-1,K1DI1
48,T01,1-201-2011-76,201,2008-11-07,0,K1DI1-101-1041-10,K1DI1
546,T05,5-201-2012-127,201,2008-11-07,0,K1DI1-101-1041-100,K1DI1
22,T01,1-201-2011-195,201,2008-11-07,0,K1DI1-101-1041-101,K1DI1
354,T02,2-201-2011-280,201,2008-11-07,0,K1DI1-101-1041-101,K1DI1


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
12,T01,1-201-2011-424,201,2008-11-07,0,K1BE2-101-1011-13,K1BE2
13,T01,1-201-2011-242,201,2008-11-07,1,K1BE2-101-1011-14,K1BE2
14,T01,1-201-2011-215,201,2008-11-07,0,K1BE2-101-1011-19,K1BE2
0,T01,1-201-2011-57,201,2008-11-07,0,K1BE2-101-1011-2,K1BE2
113,T02,2-201-2011-260,201,2008-11-07,0,K1BE2-101-1011-2,K1BE2


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
8,T01,1-201-2011-352,201,2008-11-07,0,K1DI2-102-1021-1,K1DI2
118,T02,2-201-2011-86,201,2008-11-07,1,K1DI2-102-1021-100,K1DI2
119,T02,2-201-2011-533,201,2008-11-08,0,K1DI2-102-1021-103,K1DI2
62,T01,1-201-2011-653,201,2008-11-08,0,K1DI2-102-1021-104,K1DI2
120,T02,2-201-2011-375,201,2008-11-08,0,K1DI2-102-1021-105,K1DI2


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
0,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1
1,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1
2,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
3,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1
4,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1


,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type
4995,T01,1-201-2011-536,201,2008-11-08,1,K1DI2-103-1031-85,K1DI2
4996,T01,1-201-2011-80,201,2008-11-07,0,K1DI2-103-1031-89,K1DI2
4997,T01,1-201-2011-520,201,2008-11-08,0,K1DI2-103-1031-94,K1DI2
4998,T01,1-201-2011-615,201,2008-11-08,1,K1DI2-103-1031-97,K1DI2
4999,T01,1-201-2011-718,201,2008-11-08,0,K1DI2-103-1031-99,K1DI2


looking if the engine is in oem1

In [171]:
final_df["in_ome1"] = final_df["Motor_ID"].isin(oem_combined).astype(int)


display(final_df.head())
display(final_df.isna().sum())

,Part_Type,Part_ID,Herstellernummer,Produktionsdatum,Fehlerhaft,Motor_ID,Engine Type,in_ome1
0,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1,1
1,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
2,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
3,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
4,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1,1


Part_Type           0
Part_ID             0
Herstellernummer    0
Produktionsdatum    0
Fehlerhaft          0
Motor_ID            0
Engine Type         0
in_ome1             0
dtype: int64

renaming the columns

In [172]:
final_df = final_df.rename(columns={
    "Part_Type": "part_type",
    "Part_ID" : "part_id",
    "Herstellernummer" : "manufacturer",
    "Produktionsdatum" : "production_date",
    "Fehlerhaft" : "faulty",
    "Motor_ID" : "engine_id",
    "Engine Type" : "engine_type",
    "in_ome1" : "OEM_type"
                            })

display(final_df.head())

,part_type,part_id,manufacturer,production_date,faulty,engine_id,engine_type,OEM_type
0,T01,1-201-2011-45,201,2008-11-07,1,K1BE1-101-1011-1,K1BE1,1
1,T02,2-201-2011-161,201,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
2,T03,3-202-2023-14,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
3,T04,4-202-2023-20,202,2008-11-07,0,K1BE1-101-1011-1,K1BE1,1
4,T01,1-201-2011-37,201,2008-11-07,0,K1BE1-101-1011-10,K1BE1,1


Exporting to csv

In [174]:
final_df.to_csv("final_df.csv", index=False)